In [51]:
import pandas as pd
import os
import json
from scipy.signal import savgol_filter
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm

config = json.load(open('../config.json', 'r'))
df = pd.read_parquet(os.path.join(config['data_path'], config['file_name']))

df.head()

,user_str,arrival_time,departure_time,gps_coordinates,geometry.latitude,geometry.longitude,safegraph.placekey,safegraph.place_id,safegraph.parent_placekey,safegraph.parent_place_id,...,safegraph.iso_country_code,safegraph.dist_to_poi,user_id,place_id,is_closed,category,naics_2digit,top_naics_category,place_lat,place_lon
0,000ca75d94ad7cf2dfcc7045226853eb08916fd60e3387...,2019-01-01T21:50:22Z,2019-01-01T22:28:39Z,"[-118.14454960000002, 34.6105032]",34.610503,-118.144550,None,None,None,None,...,None,NaN,17,0,False,0,None,0,34.610503,-118.144550
1,007e83a8e8413e323630553115a1fdc73e5752008a72af...,2019-01-01T00:42:05Z,2019-01-01T00:53:35Z,"[-117.97649, 33.981121]",33.981121,-117.976490,None,None,None,None,...,None,NaN,116,0,False,0,None,0,33.981121,-117.976490
2,007e83a8e8413e323630553115a1fdc73e5752008a72af...,2019-01-01T00:53:35Z,2019-01-01T01:02:04Z,"[-117.975365, 33.991165]",33.991165,-117.975365,None,None,None,None,...,None,NaN,116,0,False,0,None,0,33.991165,-117.975365
3,00a2ee076b12db94d153c47e002f256d5d51cbe25260be...,2019-01-01T01:12:29Z,2019-01-01T01:21:16Z,"[-118.28952, 34.006786]",34.006786,-118.289520,22q-222@5z6-3q7-nyv,sg:4c4eff6a70674defab02487b10848bf1,None,None,...,US,86.105909,154,12258,True,114,81,21,34.006238,-118.290180
4,00a2ee076b12db94d153c47e002f256d5d51cbe25260be...,2019-01-01T04:48:57Z,2019-01-01T05:13:05Z,"[-118.289665, 34.006255]",34.006255,-118.289665,22q-222@5z6-3q7-nyv,sg:4c4eff6a70674defab02487b10848bf1,None,None,...,US,47.509287,154,12258,True,114,81,21,34.006238,-118.290180


In [52]:
df.shape

(6908365, 45)

In [53]:
# use Houston timezone
local_tz = 'America/Chicago'
df['arrival_time'] = pd.to_datetime(df['arrival_time']) #.dt.tz_localize('UTC').dt.tz_convert(local_tz)
df['departure_time'] = pd.to_datetime(df['departure_time']) #.dt.tz_localize('UTC').dt.tz_convert(local_tz)

# df['arrival_time'] = pd.to_datetime(df['arrival_time']).dt.tz_convert(local_tz)
# df['departure_time'] = pd.to_datetime(df['departure_time']).dt.tz_convert(local_tz)

counts = df['safegraph.place_id'].value_counts()
counts = counts[counts > 500]
print(f"Number of popular POIs: {len(counts)}")

df = df[df['safegraph.place_id'].isin(counts.index)]

Number of popular POIs: 192


In [54]:
def to_day_of_week_float(series):
    dt = series.dt
    return dt.day_of_week + dt.hour / 24.0 + dt.minute / 1440.0

def to_hour_of_day_float(series):
    dt = series.dt
    return dt.hour + dt.minute / 60.0
    
def construct_popular_times(poi_visits, converter):
    df = pd.DataFrame({
        'pos': pd.concat([
            converter(poi_visits['arrival_time']),
            converter(poi_visits['departure_time']),
        ], ignore_index=True),
        'change': [1] * len(poi_visits['arrival_time']) + [-1] * len(poi_visits['departure_time']),
    }).sort_values(by='pos').reset_index(drop=True)
    df['value'] = df['change'].cumsum()
    df['value'] -= df['value'].min()
    return df

def make_smooth_y(x, df, period, window_length, poly_order):
    y = np.interp(x, df['pos'], df['value'], period=period)
    y_smoothed = savgol_filter(y, window_length, poly_order, mode='wrap')
    return y_smoothed

resolution = 100
num_decimals = 4
hod_x = np.linspace(0, 24, resolution)
dow_x = np.linspace(0, 7, resolution)

rows = []
for poi_id, poi_visits in tqdm(df.groupby('safegraph.place_id')):
    num_visits = len(poi_visits)

    hour_of_day_changes = construct_popular_times(poi_visits, to_hour_of_day_float)
    hod_y = make_smooth_y(hod_x, hour_of_day_changes, period=24, window_length=10, poly_order=3)
    hod_y /= hod_y.sum()
    hod_y = np.round(hod_y, num_decimals).tolist()

    day_of_week_changes = construct_popular_times(poi_visits, to_day_of_week_float)
    dow_y = make_smooth_y(dow_x, day_of_week_changes, period=7, window_length=10, poly_order=3)
    dow_y /= dow_y.sum()
    dow_y = np.round(dow_y, num_decimals).tolist()
    
    rows.append({
        'safegraph_place_id': poi_id,
        'num_visits': num_visits,
        'daily': hod_y,
        'weekly': dow_y,
    })

  0%|          | 0/192 [00:00<?, ?it/s]

100%|██████████| 192/192 [00:01<00:00, 155.69it/s]


In [55]:
new_df = pd.DataFrame(rows)
new_df.to_parquet('/data/private/maria_data/data/cache/LosAngeles/popular_times_500.parquet', index=False, engine='pyarrow')